# AI vs Otsu: all cell types in Lung and Lesion

This notebook combines all `Lung` regions and all `Lesion` regions in each project. It counts every cell type and compares its Lesion-versus-Lung odds ratio for AI and Otsu positivity.

No correction is added to zero counts. Consequently, an odds ratio can be `0`, infinite, or undefined.

In [ ]:
from pathlib import Path
import gc
import json
import sys

import numpy as np
import pandas as pd
from scipy import ndimage
from skimage.draw import polygon

REPO = Path(r"C:/opal-studio")
PROJECTS = REPO / "Projects"
SEGMENTATION = "StarDist_Expand"
MEMBERSHIP_FRACTION = 0.5

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from opal_studio import project_io

In [ ]:
def project_details(path):
    parts = path.stem.rsplit("_", 2)
    return parts[-2], parts[-1]  # e.g. Q001, AI


def load_project(path):
    if not (path / "zarr.json").exists():
        raise FileNotFoundError(f"Not an Opal Studio project: {path}")
    return project_io.load_project(path)


def stored_cell_types(path):
    metadata = json.loads((path / "zarr.json").read_text(encoding="utf-8"))
    channels = metadata["attributes"]["opal_studio"]["session"]["channels"]
    return [channel["name"] for channel in channels if channel.get("kind") == "type_mask"]


def region_names(doc, group):
    """Find Region, Region 1, Region 2, and so on."""
    group = group.casefold()
    names = [
        name for name in doc.shapes
        if name.casefold() == group or name.casefold().startswith(group + " ")
    ]
    if not names:
        raise KeyError(f"No {group!r} region found; available regions: {sorted(doc.shapes)}")
    return names


def combined_region_mask(doc, group, shape):
    """Rasterise and combine every region belonging to one group."""
    mask = np.zeros(shape, dtype=bool)
    for name in region_names(doc, group):
        for rings in doc.shapes[name].values():
            for ring in rings:
                xy = np.asarray(ring, dtype=float)
                if len(xy) < 3:
                    continue
                rows, columns = polygon(xy[:, 1], xy[:, 0], shape)
                mask[rows, columns] = True
    return mask


def typed_cells(cell_labels, type_mask, cell_ids):
    """Return True for cells with at least 50% of their pixels in a type mask."""
    cell_sizes = np.bincount(cell_labels.ravel(), minlength=int(cell_labels.max()) + 1)
    covered = np.bincount(
        cell_labels[type_mask > 0].ravel(), minlength=cell_sizes.size
    )
    fractions = covered / np.maximum(cell_sizes, 1)
    return fractions[cell_ids] >= MEMBERSHIP_FRACTION


def analyse_project(path, cell_types):
    doc = load_project(path)
    sample, method = project_details(path)

    cell_labels = np.asarray(doc.labels[SEGMENTATION], dtype=np.int32)
    cell_ids = np.unique(cell_labels)
    cell_ids = cell_ids[cell_ids > 0]

    centres = np.asarray(ndimage.center_of_mass(cell_labels > 0, cell_labels, cell_ids))
    centre_y = np.clip(centres[:, 0].astype(int), 0, cell_labels.shape[0] - 1)
    centre_x = np.clip(centres[:, 1].astype(int), 0, cell_labels.shape[1] - 1)

    in_lung = combined_region_mask(doc, "Lung", cell_labels.shape)[centre_y, centre_x]
    in_lesion = combined_region_mask(doc, "Lesion", cell_labels.shape)[centre_y, centre_x]

    # Lesion takes precedence if drawn regions overlap.
    regions = np.full(cell_ids.size, "Outside", dtype=object)
    regions[in_lung] = "Lung"
    regions[in_lesion] = "Lesion"

    rows = []
    for cell_type in cell_types:
        if cell_type in doc.labels:
            is_type = typed_cells(cell_labels, doc.labels[cell_type], cell_ids)
        else:
            is_type = np.zeros(cell_ids.size, dtype=bool)
        for region in ["Lesion", "Lung"]:
            in_region = regions == region
            total = int(in_region.sum())
            count = int((is_type & in_region).sum())
            rows.append({
                "Sample": sample,
                "Method": method,
                "Cell type": cell_type,
                "Region": region,
                "Target cells": count,
                "Total cells": total,
                "Percent": 100 * count / total if total else np.nan,
            })
    return pd.DataFrame(rows)

In [ ]:
def odds_ratio(lesion_cells, lesion_total, lung_cells, lung_total):
    """Lesion-vs-Lung odds ratio, without a zero-count correction."""
    numerator = lesion_cells * (lung_total - lung_cells)
    denominator = (lesion_total - lesion_cells) * lung_cells

    if denominator == 0:
        return np.inf if numerator > 0 else np.nan
    return numerator / denominator


def comparison_table(counts):
    rows = []
    groups = counts.groupby(["Sample", "Method", "Cell type"], sort=False)
    for (sample, method, cell_type), group in groups:
        lesion = group.set_index("Region").loc["Lesion"]
        lung = group.set_index("Region").loc["Lung"]
        ratio = odds_ratio(
            lesion["Target cells"], lesion["Total cells"],
            lung["Target cells"], lung["Total cells"],
        )
        with np.errstate(divide="ignore", invalid="ignore"):
            log2_ratio = np.log2(ratio)

        rows.append({
            "Sample": sample,
            "Method": method,
            "Cell type": cell_type,
            "Lesion cells": int(lesion["Target cells"]),
            "Lesion total": int(lesion["Total cells"]),
            "Lesion %": lesion["Percent"],
            "Lung cells": int(lung["Target cells"]),
            "Lung total": int(lung["Total cells"]),
            "Lung %": lung["Percent"],
            "Odds ratio": ratio,
            "log2 odds ratio": log2_ratio,
        })
    return pd.DataFrame(rows)

## Results

An odds ratio above 1 (and a log2 odds ratio above 0) means the cell type is enriched in Lesion relative to Lung.

In [ ]:
project_paths = [
    path for path in sorted(PROJECTS.glob("*.zarr"))
    if path.stem.endswith(("_AI", "_Otsu"))
]

types_by_project = {path: stored_cell_types(path) for path in project_paths}
cell_types = []
for names in types_by_project.values():
    cell_types.extend(name for name in names if name not in cell_types)

print(f"Found {len(cell_types)} cell types across {len(project_paths)} projects.")
for path, names in types_by_project.items():
    missing = [name for name in cell_types if name not in names]
    if missing:
        print(f"{path.name}: no stored cells for {', '.join(missing)}; counted as zero")

project_counts = []
for path in project_paths:
    print(f"Reading {path.name}")
    project_counts.append(analyse_project(path, cell_types))
    gc.collect()

counts = pd.concat(project_counts, ignore_index=True)
results = comparison_table(counts).sort_values(
    ["Cell type", "Sample", "Method"]
).reset_index(drop=True)

display(results.round({
    "Lesion %": 2,
    "Lung %": 2,
    "Odds ratio": 3,
    "log2 odds ratio": 3,
}))

In [ ]:
median_results = (
    results.groupby(["Cell type", "Method"])[
        ["Lesion %", "Lung %", "Odds ratio", "log2 odds ratio"]
    ]
    .median()
    .round(3)
)
display(median_results)

## Overall cell-type composition

Total variation and Jensen–Shannon distance compare the complete cell-type distributions in Lesion and Lung. Both range from 0 (identical compositions) to 1 (completely separate compositions), handle zero counts without a correction, and do not assume that every cell type should be enriched in Lesion.

In [ ]:
def composition_separation(counts):
    rows = []
    for (sample, method), group in counts.groupby(["Sample", "Method"]):
        table = (
            group.pivot(index="Cell type", columns="Region", values="Target cells")
            .reindex(cell_types)
            .fillna(0)
        )
        lesion_counts = table["Lesion"].to_numpy(dtype=float)
        lung_counts = table["Lung"].to_numpy(dtype=float)
        lesion_distribution = lesion_counts / lesion_counts.sum()
        lung_distribution = lung_counts / lung_counts.sum()

        total_variation = 0.5 * np.abs(
            lesion_distribution - lung_distribution
        ).sum()

        midpoint = 0.5 * (lesion_distribution + lung_distribution)
        lesion_nonzero = lesion_distribution > 0
        lung_nonzero = lung_distribution > 0
        js_divergence = 0.5 * np.sum(
            lesion_distribution[lesion_nonzero]
            * np.log2(lesion_distribution[lesion_nonzero] / midpoint[lesion_nonzero])
        ) + 0.5 * np.sum(
            lung_distribution[lung_nonzero]
            * np.log2(lung_distribution[lung_nonzero] / midpoint[lung_nonzero])
        )

        lesion_total = group.loc[group["Region"] == "Lesion", "Total cells"].iloc[0]
        lung_total = group.loc[group["Region"] == "Lung", "Total cells"].iloc[0]
        rows.append({
            "Sample": sample,
            "Method": method,
            "Lesion assigned %": 100 * lesion_counts.sum() / lesion_total,
            "Lung assigned %": 100 * lung_counts.sum() / lung_total,
            "Total variation": total_variation,
            "Jensen-Shannon distance": np.sqrt(js_divergence),
        })
    return pd.DataFrame(rows)


overall = composition_separation(counts)
display(overall.round(3))
display(
    overall.groupby("Method")[
        ["Lesion assigned %", "Lung assigned %",
         "Total variation", "Jensen-Shannon distance"]
    ].median().round(3)
)